# Đề kiểm tra giữa kỳ

Họ và tên: Nguyễn Phúc Khang

Lớp: 22127180

## Mô tả dữ liệu

Tập Dữ liệu được sử dụng trong bài tập là **Student Performance Factors** được lấy từ [Kaggle](https://www.kaggle.com/datasets/lainguyn123/student-performance-factors). Tập dữ liệu cho biết một số nhân tố ảnh hưởng đến điểm số của học sinh.
Tập dữ liệu sau khi được xử lý gồm *6378 dòng* và *20 cột* với ý nghĩa từng cột như sau:


| Thuộc tính                 | Mô tả                                                                         |
|----------------------------|-------------------------------------------------------------------------------|
| Hours_Studied              | Số giờ học mỗi tuần.                                                          |
| Attendance                 | Tỷ lệ phần trăm số buổi học đã tham gia.                                      |
| Parental_Involvement       | Mức độ tham gia của phụ huynh vào việc học của học sinh (Low, Medium, High).  |
| Access_to_Resources        | Mức độ sẵn có của các tài nguyên giáo dục (Low, Medium, High).                |
| Extracurricular_Activities | Tham gia vào các hoạt động ngoại khóa (Yes, No).                              |
| Sleep_Hours                | Số giờ ngủ trung bình mỗi đêm.                                                |
| Previous_Scores            | Điểm số từ các kỳ thi trước.                                                  |
| Motivation_Level           | Mức độ động lực học tập của học sinh (Low, Medium, High).                     |
| Internet_Access            | Mức độ truy cập Internet (Yes, No).                                           |
| Tutoring_Sessions          | Số buổi học phụ đạo tham gia mỗi tháng.                                       |
| Family_Income              | Mức thu nhập của gia đình (Low, Medium, High).                                |
| Teacher_Quality            | Chất lượng giáo viên (Low, Medium, High).                                     |
| School_Type                | Loại hình trường học (Public, Private).                                       |
| Peer_Influence             | Ảnh hưởng của bạn bè đến kết quả học tập (Positive, Neutral, Negative).       |
| Physical_Activity          | Số giờ hoạt động thể chất trung bình mỗi tuần.                                |
| Learning_Disabilities      | Có sự hiện diện của khuyết tật học tập hay không (Yes, No).                   |
| Parental_Education_Level   | Trình độ học vấn cao nhất của cha mẹ (High School, College, Postgraduate).    |
| Distance_from_Home         | Khoảng cách từ nhà đến trường (Near, Moderate, Far).                          |
| Gender                     | Giới tính của học sinh (Male, Female).                                        |
| Exam_Score                 | Điểm kỳ thi cuối cùng.                                                        |

Tập dữ liệu sẽ được chia thành hai tệp tin là: `train.csv` và `test.csv` với số lượng bản ghi tương ứng là  5102 và 1276.

## Yêu cầu đề bài

Trong nội dung kiểm tra này các bạn sẽ xây dựng hai mô hình Decision Tree và mô hình Linear Regression để dự đoán cột điểm số `Exam_score` dựa trên các thuộc tính còn lại.

**Yêu cầu chung**
* Các hành vi gian lận sẽ bị đánh giá **0 điểm** trong phần giữa kỳ.

* Chỉ được phép sử dụng các thư viện `pandas` và `numpy`. Các bạn có thể sử dụng thêm các thư viện trực quan như `seaborn` hoặc `matplotlib` để trực quan dữ liệu.

* Dữ liệu huấn luyện mô hình sẽ lấy từ file `train.csv`

* Dữ liệu kiểm tra mô hình các bạn sẽ lấy từ file `test.csv`

* **Đối với mô hình Desion Tree**, dữ liệu chỉ gồm các cột `Parental_Involvement`, `Access_to_Resources`, `Extracurricular_Activities`, `Motivation_Level`, `Internet_Access`, `Family_Income`, `Teacher_Quality`, `School_Type`, `Peer_Influence`, `Learning_Disabilities`, `Parental_Education_Level`, `Distance_from_Home`, `Gender` và cột điểm số `Exam_score`

* **Đối với mô hình Linear Regression**, dữ liệu chỉ gồm các cột `Hours_Studied`, `Attendance`, `Sleep_Hours`, `Previous_Scores`, `Tutoring_Sessions`, `Physical_Activity` và cột điểm số `Exam_score`.

### Thư viện

In [24]:
import numpy as np
import pandas as pd

### Đọc file

In [25]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

### Câu 1 - Decision Tree

a. Viết chương trình xây dựng cây quyết định (theo mẫu) trên các tất cả thuộc tính (trừ `Exam_Score`). Xác định độ lỗi MSE (mean square error) của mô hình trên tập test.

b. Xây dựng 5 mô hình cây quyết định bằng việc chọn lần lượt 1 thuộc tính, 2 thuộc tính, 3 thuộc tính, 4 thuộc tính, 5 thuộc tính. Giải thích lý do lựa chọn các cột đó và tính toán độ lỗi MSE trên tập test.

#### a.1 Xây dựng class Decision Tree

In [26]:

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=10):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y):
        """Fit the decision tree to the data."""
        data = pd.concat([X, y], axis=1)
        self.tree = self._build_tree(data, depth=0)
        
    def _build_tree(self, data, depth):
        X = data.iloc[:, :-1]
        y = data.iloc[:, -1]
        
        #  Điều kiện dừng
        if len(y) < self.min_samples_split or depth >= self.max_depth:
            return self._leaf(y)
        
        # Tìm feature và value tốt nhất để split
        best_feature, best_value = self._best_split(X, y)
        if best_feature is None:
            return self._leaf(y)

        # Khởi tạo node trái và phải
        left_data = data[data[best_feature] <= best_value]
        right_data = data[data[best_feature] > best_value]
        
        return {
            'feature': best_feature,
            'value': best_value,
            'left': self._build_tree(left_data, depth + 1),
            'right': self._build_tree(right_data, depth + 1)
        }

    def _leaf(self, y):
        return np.mean(y)

    def _best_split(self, X, y):
        best_mse = float('inf')
        best_feature = None
        best_value = None
        
        for feature in X.columns:
            values = X[feature].unique()
            for value in values:
                left_y = y[X[feature] <= value]
                right_y = y[X[feature] > value]
                
                mse = self._mse(left_y, right_y)
                if mse < best_mse:
                    best_mse = mse
                    best_feature = feature
                    best_value = value

        return best_feature, best_value

    def _mse(self, left_y, right_y):
        def mse(y):
            return np.mean((y - np.mean(y)) ** 2) if len(y) > 0 else 0
        
        total = len(left_y) + len(right_y)
        if total == 0:
            return 0
        return (len(left_y) / total) * mse(left_y) + (len(right_y) / total) * mse(right_y)

    def predict(self, X):
        """Predict the target value for each instance in X."""
        predictions = [self._predict_instance(instance, self.tree) for _, instance in X.iterrows()]
        return np.array(predictions)

    def _predict_instance(self, instance, node):
        if isinstance(node, dict):
            if instance[node['feature']] <= node['value']:
                return self._predict_instance(instance, node['left'])
            else:
                return self._predict_instance(instance, node['right'])
        else:
            return node

    def loss(self, y_pred, y):
        """Calculate Mean Squared Error loss between predictions and true values."""
        return np.mean((y_pred - y) ** 2)

    def drawTree(self, node=None, depth=0):
        """Text representation of the tree structure."""
        if node is None:
            node = self.tree

        if isinstance(node, dict):
            print(" " * 4 * depth + f"[{node['feature']}] <= {node['value']}")
            self.drawTree(node['left'], depth + 1)
            self.drawTree(node['right'], depth + 1)
        else:
            print(" " * 4 * depth + f"Leaf: {node}")

#### a.2 Huấn luyện mô hình

In [27]:
features = ['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 
            'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 
            'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 
            'Distance_from_Home', 'Gender']
X_train = train[features]
y_train = train['Exam_Score']

# Khởi tạo mô hình DecisionTree
decision_tree = DecisionTree(max_depth=5)
decision_tree.fit(X_train, y_train)

# Dự đoán kết quả trên tập test
X_test = test[features]
y_test = test['Exam_Score']
y_pred = decision_tree.predict(X_test)

# Tính độ lỗi MES
loss = decision_tree.loss(y_pred, y_test)
print(f"MSE Loss: {loss}")

# vẽ cây quyết định
decision_tree.drawTree()

MSE Loss: 14.672105909900587
[Access_to_Resources] <= High
    [Parental_Involvement] <= High
        [Family_Income] <= Low
            [Family_Income] <= High
                [Extracurricular_Activities] <= No
                    Leaf: 68.15384615384616
                    Leaf: 70.76470588235294
                [Distance_from_Home] <= Moderate
                    Leaf: 67.02898550724638
                    Leaf: 68.70707070707071
            [Teacher_Quality] <= High
                [Peer_Influence] <= Neutral
                    Leaf: 72.17857142857143
                    Leaf: 69.72222222222223
                [Motivation_Level] <= Low
                    Leaf: 70.0
                    Leaf: 68.92105263157895
        [Peer_Influence] <= Negative
            [Learning_Disabilities] <= No
                [Motivation_Level] <= High
                    Leaf: 68.7948717948718
                    Leaf: 66.63583815028902
                [Motivation_Level] <= High
                    Leaf

#### b.1 Xây dựng mô hình

In [28]:
def evaluate_model(selected_features):
    X_train_subset = X_train[selected_features]
    X_test_subset = X_test[selected_features]
    
    decision_tree = DecisionTree(max_depth=5)
    decision_tree.fit(X_train_subset, y_train)
    y_pred_subset = decision_tree.predict(X_test_subset)
    
    mse_loss_subset = decision_tree.loss(y_pred_subset, y_test)
    return mse_loss_subset

feature_combinations = [
    ['Parental_Involvement'],
    ['Motivation_Level', 'Extracurricular_Activities'], 
    ['Access_to_Resources', 'Internet_Access', 'Family_Income'], 
    ['Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities'], 
    ['Parental_Education_Level', 'Parental_Involvement', 'Motivation_Level', 'Distance_from_Home', 'Access_to_Resources'] 
]

for i, features in enumerate(feature_combinations, start=1):
    mse = evaluate_model(features)
    print(f"MSE Loss Model {i}: {mse} {features}")


MSE Loss Model 1: 15.104421480932084 ['Parental_Involvement']
MSE Loss Model 2: 15.27210150357797 ['Motivation_Level', 'Extracurricular_Activities']
MSE Loss Model 3: 14.979867105573828 ['Access_to_Resources', 'Internet_Access', 'Family_Income']
MSE Loss Model 4: 15.409879923544485 ['Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities']
MSE Loss Model 5: 14.582847985178892 ['Parental_Education_Level', 'Parental_Involvement', 'Motivation_Level', 'Distance_from_Home', 'Access_to_Resources']


#### b.2 Giải thích

##### Mô hình 1: `Parental_Involvement`
- **Parental_Involvement**: Mức độ hỗ trợ từ gia đình có ảnh hưởng lớn đến kết quả học tập của học sinh, giúp học sinh có động lực và định hướng rõ ràng trong học tập.

##### Mô hình 2: `Motivation_Level`, `Extracurricular_Activities`
- **Motivation_Level**: Học sinh có động lực cao thường có khả năng tập trung tốt, chú ý hơn trong lớp học, hoàn thành bài tập, và tham gia tích cực vào các hoạt động học tập. Có khả năng vượt qua khó khăn và cảm giác chán nản trong quá trình học tập tốt hơn.
- **Extracurricular_Activities**: Hỗ trợ phát triển kỹ năng xã hội và tạo môi trường học tập tích cực.

##### Mô hình 3: `Access_to_Resources`, `Internet_Access`, `Family_Income`
- **Access_to_Resources**, **Internet_Access**: Truy cập vào tài liệu học tập và Internet giúp học sinh tiếp thu kiến thức tốt hơn. Thiếu tài nguyên có thể dẫn đến khó khăn trong việc học.
- **Family_Income**: Gia đình có thu nhập cao hơn thường có khả năng cung cấp cho con cái nhiều tài nguyên học tập hơn, như sách vở, công nghệ (máy tính, internet), và các lớp học thêm. Ngoài ra còn giảm bớt áp lực tài chính để con cái có khả cơ hội tập trung chuyên sâu về việc học hơn.

##### Mô hình 4: `Teacher_Quality`, `School_Type`, `Peer_Influence`, và `Learning_Disabilities`
- **Teacher_Quality**: Ảnh hưởng chất lượng bài giảng, khả năng truyền đạt có giúp học sinh tiếp thu tốt hay không.
- **School_Type**: Môi trường học, cơ sở vật chất tốt tạo điều kiện cho học sinh học sinh có tinh thần thoải mái, đầy đủ thiết bị học tập.
- **Peer_Influence**: Bạn bè cũng là yếu tố rất quan trọng, bạn bè có thể hỗ trợ giúp đỡ lần nhau, chia sẽ kiến thức trong quá trình học tập.
= **Learning_Disabilities**: Gây khó khăn trong việc tiếp thu hoặc xử lý thông tin, ảnh hưởng đến khả năng học tập của học sinh. Điều này có thể bao gồm các vấn đề về đọc (dyslexia), viết (dysgraphia), hoặc toán học (dyscalculia).

##### Mô hình 5: `Parental_Education_Level`, `Parental_Involvement`, `Motivation_Level`, `Distance_from_Home`, `Access_to_Resources`
- **Parental_Education_Level**: Theo nghiên cứu cho thấy trẻ em có cha mẹ có trình độ học vấn cao thường có kết quả học tập tốt hơn. Cha mẹ trình độ học vấn cao, có thể cung cấp một môi trường học tập hỗ trợ tốt, hướng dẫn con cái về học tập, định hướng việc học tốt hơn. Cha mẹ sẽ có xu hướng đặt giá trị cao vào giáo dục, điều này có thể truyền cảm hứng cho trẻ em nỗ lực học tập và phát triển kỹ năng cần thiết.
- **Parental_Involvement**: Sự tham gia của cha mẹ có thể bao gồm việc giúp đỡ làm bài tập, tham gia các buổi họp phụ huynh, và khuyến khích tham gia các hoạt động học tập.
- **Distance_from_Home**:  Khoảng cách xa có thể gây khó khăn cho việc tham gia đầy đủ vào các hoạt động học tập, dẫn đến việc học sinh có thể bỏ lỡ các cơ hội học tập quan trọng. Ngoài ra còn tốn nhiều thời gian và chi phí, ảnh hưởng đến thời gian mà học sinh có thể dành cho việc học và các hoạt động khác.
- **Family_Income**: Như đã giải thích ở trên.
- **Motivation_Level**: Như đã giải thích ở trên.


### Câu 2 - Linear Regression

a. Xác định mô hình hồi quy tuyến tính $y = w_0 + w_1 * \text{Attendance} + w_2 * \text{Hours-Studied} + w_3 * \text{Previous-Scores}^2$ và độ lỗi mean square error (MSE) của mô hình.

b. Xây dựng 5 mô hình hồi quy tuyến tính sử dụng ít nhất 2 thuộc tính và giải thích lý do chọn mô hình hồi quy tuyến tính. Tính độ lỗi MSE của từng mô hình.

#### Xây dựng class Linear Regression

In [29]:
class LinearRegression:
    def __init__(self):
        self.weights = None

    def fit(self, X, y):
        self.weights = np.linalg.inv(X.T @ X) @ X.T @ y

    def predict(self, X):
        return np.array(np.array(X) @ self.weights)
    
    def get_params(self):
        return self.weights
    
def preprocess(x):
    return np.hstack((np.ones((x.shape[0], 1)), x))

def mse(y, y_true):
    return np.mean((y - y_true) ** 2)


#### a.1 Chuẩn bị dữ liệu

In [30]:
X_train = np.column_stack([
    train['Attendance'],
    train['Hours_Studied'],
    train['Previous_Scores'] ** 2
])
y_train = train['Exam_Score'].values

X_test = np.column_stack([
    test['Attendance'].values,
    test['Hours_Studied'].values,
    test['Previous_Scores'].values ** 2
])
y_test = test['Exam_Score'].values

X_train_ppc = preprocess(X_train)
X_test_ppc = preprocess(X_test)

#### a.2 Huấn luyện mô hình

In [31]:
# Xác định tham số mô hình
model = LinearRegression()
model.fit(X_train_ppc, y_train)
model.get_params()

array([4.37766491e+01, 1.98094634e-01, 2.91189129e-01, 3.09623482e-04])

#### a.3 Dự đoán và tính độ lỗi

In [32]:
y_hat = model.predict(X_test_ppc)
mse_value = mse(y_hat, y_test)
print(mse_value)

6.4693511759601545


#### b.1 Xây dựng mô hình

In [33]:
list_models = {
    'Model 1': ['Hours_Studied', 'Previous_Scores'],
    'Model 2': ['Hours_Studied', 'Previous_Scores', 'Sleep_Hours'],
    'Model 3': ['Hours_Studied', 'Previous_Scores', 'Attendance'],
    'Model 4': ['Hours_Studied', 'Previous_Scores', 'Tutoring_Sessions'],
    'Model 5': ['Hours_Studied', 'Previous_Scores', 'Attendance', 'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity']
}

#### b.2 Huấn luyện và tính độ lỗi từng mô hình

In [34]:
for model_name, cols in list_models.items():
    X_train_ppc = preprocess(train[cols].values)
    X_test_ppc = preprocess(test[cols].values)

    # Khởi tạo mô hình cho từng model
    model = LinearRegression()
    model.fit(X_train_ppc, y_train)

    # Dự đoán và tính độ lỗi MSE
    y_hat = model.predict(X_test_ppc)
    mse_value = mse(y_hat, y_test)

    # Hiển thị kết quả
    print(f"{model_name}: {cols}")
    print(f"\tMSE: {mse_value}")
    print()

Model 1: ['Hours_Studied', 'Previous_Scores']
	MSE: 11.8013234451974

Model 2: ['Hours_Studied', 'Previous_Scores', 'Sleep_Hours']
	MSE: 11.81528823561214

Model 3: ['Hours_Studied', 'Previous_Scores', 'Attendance']
	MSE: 6.4780814578585035

Model 4: ['Hours_Studied', 'Previous_Scores', 'Tutoring_Sessions']
	MSE: 11.31142744951979

Model 5: ['Hours_Studied', 'Previous_Scores', 'Attendance', 'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity']
	MSE: 6.063230353809079



#### b.3 Giải thích mô hình

- **Model 1:** `['Hours_Studied', 'Previous_Scores']`  
  - Mô hình này chỉ sử dụng hai đặc trưng cơ bản là `Hours_Studied` (số giờ học) và `Previous_Scores` (điểm số trước đó). Đây là hai yếu tố ảnh hưởng rõ ràng đến kết quả thi. Tuy nhiên, MSE tương đối cao (11.80) cho thấy cần thêm yếu tố khác để cải thiện độ chính xác.

- **Model 2:** `['Hours_Studied', 'Previous_Scores', 'Sleep_Hours']`  
  - Mô hình này bổ sung thêm `Sleep_Hours` (số giờ ngủ) để xem xét ảnh hưởng của giấc ngủ đến hiệu suất học tập. MSE chỉ giảm nhẹ (11.81), cho thấy Sleep_Hours không có tác động mạnh trong mô hình này.

- **Model 3:** `['Hours_Studied', 'Previous_Scores', 'Attendance']`  
  - Việc thêm `Attendance` (sự tham dự lớp học) vào mô hình cho thấy rằng tham dự đều đặn có thể tăng khả năng tiếp thu kiến thức. MSE giảm rõ rệt xuống (6.48), cho thấy Attendance là yếu tố quan trọng giúp cải thiện độ chính xác của mô hình.

- **Model 4:** `['Hours_Studied', 'Previous_Scores', 'Tutoring_Sessions']`  
  - Mô hình này đánh giá hiệu quả của các buổi học thêm qua `Tutoring_Sessions`. Mặc dù MSE (11.31) thấp hơn một chút so với Model 1 và 2, nhưng không đủ để cải thiện đáng kể độ chính xác.

- **Model 5:** `['Hours_Studied', 'Previous_Scores', 'Attendance', 'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity']`  
  - Mô hình này bao gồm nhiều yếu tố hơn, như `Physical_Activity` (hoạt động thể chất), cùng với các yếu tố trước đó. Đây là mô hình có MSE thấp nhất (6.06), cho thấy tác động đồng thời của các yếu tố liên quan đến sức khỏe, sự tham dự và giấc ngủ có thể cải thiện đáng kể dự đoán điểm số.
